## 1. Import


In [1]:
"""
Yelp HC Feature MLP Classifier — Grid Search + Multi-Seed
==========================================================
Data   : data_yelp.parquet
Input  : 23개 HC features  (log(x+1) 스케일링)
         ※ LIWC 관련 5개 제거:
            anger, sadness, posemo, anx, negate
Model  : MLP  23 → d_model → d_model → d_model → 1 (Sigmoid)
         Optimizer : Adam (weight_decay=1e-4)
         Loss      : BCELoss
         Scheduler : ReduceLROnPlateau (val_loss 기준)

HC Features (23개):
    sentiment, subjectivity,
  텍스트 기본 (7) : syllable, lexicon, sentence, char, letter,
                    polysyllab, monosyllab
  가독성      (6) : smog_index, flesch_reading_ease, flesch_kincaid_grade,
                    fog_scale, dale_chall, reading_time
  LLM 관련   (2) : perplexity, burstiness
  품사       (6) : nouns, adj, verbs, pronoun, adverb, article

Grid   : LR × Dropout × d_model × Batch = 81 combinations
Early  : val_loss 기준, patience=5, 최대 30 epoch
Seeds  : 42, 43, 44, 45, 46
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    classification_report, confusion_matrix
)
from itertools import product
import warnings
warnings.filterwarnings("ignore")


## 2. 설정

In [ ]:

# ══════════════════════════════════════════════════════
# 0. 설정
# ══════════════════════════════════════════════════════
PARQUET_PATH   = "../data/data_yelp.parquet"
LABEL_COL      = "label"
EPOCHS         = 30
EARLY_STOP_PAT = 5

SEED_LIST    = [42, 43, 44, 45, 46]

LR_GRID      = [3e-4] # 3e-5, 1e-4, 
DROPOUT_GRID = [0.3] # 0.0, 0.1, 
DMODEL_GRID  = [512] # 128, 256, 
BATCH_GRID   = [64] # 32, 128

# ── LIWC 관련 제거 목록 ────────────────────────────────
LIWC_FEATURES = [
    "anger", "sadness", "posemo", "anx", "negate",
]

HC_FEATURES = [
    "sentiment", "subjectivity",
    # 텍스트 기본 통계 (7)
    "syllable", "lexicon", "sentence", "char", "letter",
    "polysyllab", "monosyllab",
    # 가독성 지표 (6)
    "smog_index", "flesch_reading_ease", "flesch_kincaid_grade",
    "fog_scale", "dale_chall", "reading_time",
    # LLM 관련 (2)
    "perplexity", "burstiness",
    # 품사 (6)
    "nouns", "adj", "verbs", "pronoun", "adverb", "article",
]
INPUT_DIM = len(HC_FEATURES)   # 23

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device  : {device}")
print(f"Input dim     : {INPUT_DIM}  (LIWC 5개 제거: {LIWC_FEATURES})")
print(f"Seeds         : {SEED_LIST}")
print(f"Grid combos   : {len(LR_GRID)*len(DROPOUT_GRID)*len(DMODEL_GRID)*len(BATCH_GRID)}")


Using device  : cuda
Input dim     : 23  (LIWC 5개 제거: ['anger', 'sadness', 'posemo', 'anx', 'negate'])
Seeds         : [42, 43, 44, 45, 46]
Grid combos   : 1


## 3. 데이터 로드 및 스케일링


In [3]:
# ══════════════════════════════════════════════════════
# 1. 데이터 로드
# ══════════════════════════════════════════════════════
print("\n[1] Loading data...")
df = pd.read_parquet(PARQUET_PATH)
print(f"  Shape   : {df.shape}")
print(f"  Label   : {df[LABEL_COL].value_counts().to_dict()}")

unique_labels = sorted(df[LABEL_COL].unique())
label_map = {unique_labels[0]: 0, unique_labels[1]: 1}
print(f"  Label map : {label_map}  (0='{unique_labels[0]}' / 1='{unique_labels[1]}')")
df["y"] = df[LABEL_COL].map(label_map).astype(float)

df[HC_FEATURES] = df[HC_FEATURES].fillna(df[HC_FEATURES].median())

print(f"\n  ── 원본 HC Feature 통계 (21개) ─────────────────────────")
print(df[HC_FEATURES].describe().round(3).to_string())

# ══════════════════════════════════════════════════════
# 2. log(x+1) 스케일링
# ══════════════════════════════════════════════════════
def log1p_scale(arr: np.ndarray) -> np.ndarray:
    """log(max(x, 0) + 1) — 음수값은 0으로 클리핑 후 변환"""
    return np.log1p(np.clip(arr, a_min=0, a_max=None))

X_raw = df[HC_FEATURES].values.astype(np.float32)
X_all = log1p_scale(X_raw)
y_all = df["y"].values.astype(np.float32)

print(f"\n  ── log(x+1) 변환 후 통계 ───────────────────────────────")
print(pd.DataFrame(X_all, columns=HC_FEATURES).describe().round(3).to_string())


[1] Loading data...
  Shape   : (20000, 35)
  Label   : {'human': 10000, 'ai': 10000}
  Label map : {'ai': 0, 'human': 1}  (0='ai' / 1='human')

  ── 원본 HC Feature 통계 (21개) ─────────────────────────
       sentiment  subjectivity   syllable    lexicon   sentence       char     letter  polysyllab  monosyllab  smog_index  flesch_reading_ease  flesch_kincaid_grade  fog_scale  dale_chall  reading_time  perplexity  burstiness      nouns        adj      verbs    pronoun     adverb    article
count  20000.000     20000.000  20000.000  20000.000  20000.000  20000.000  20000.000   20000.000   20000.000   20000.000            20000.000             20000.000  20000.000   20000.000     20000.000   20000.000   20000.000  20000.000  20000.000  20000.000  20000.000  20000.000  20000.000
mean       0.170         0.559    128.521     88.524      6.551    408.411    393.162       9.175      61.038       9.834               68.786                 7.100      9.395       8.422         6.000     131.013   

## 4. 모델

In [4]:
# ══════════════════════════════════════════════════════
# 3. Dataset
# ══════════════════════════════════════════════════════
class HCDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# ══════════════════════════════════════════════════════
# 4. 모델  ( 23 → d_model × 3 → 1 )
# ══════════════════════════════════════════════════════
class HCMLP(nn.Module):
    """
    MLP: INPUT_DIM → d_model → d_model → d_model → 1 (Sigmoid)
    블록: Linear → BatchNorm → ReLU → Dropout
    """
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, d_model),
            nn.BatchNorm1d(d_model), nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(d_model, d_model),
            nn.BatchNorm1d(d_model), nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(d_model, d_model),
            nn.BatchNorm1d(d_model), nn.ReLU(), nn.Dropout(dropout),

            nn.Linear(d_model, 1),
            nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x)


## 5. 학습 함수

In [5]:
# ══════════════════════════════════════════════════════
# 5. 학습 함수
# ══════════════════════════════════════════════════════
criterion = nn.BCELoss()

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, preds, labels = 0.0, [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out  = model(X_b)
            loss = criterion(out, y_b)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(X_b)
            preds.extend(out.squeeze().cpu().detach().tolist())
            labels.extend(y_b.squeeze().cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    pred_bin = (np.array(preds) >= 0.5).astype(int)
    return avg_loss, accuracy_score(labels, pred_bin), f1_score(labels, pred_bin, zero_division=0)


def train_single(lr, dropout, d_model, batch_size,
                 X_tr, y_tr, X_vl, y_vl, verbose=False):
    train_loader = DataLoader(HCDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(HCDataset(X_vl, y_vl), batch_size=batch_size)

    model     = HCMLP(d_model=d_model, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5
    )

    best_val_loss, best_val_f1 = float("inf"), 0.0
    best_state, no_improve = None, 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, _     = run_epoch(model, train_loader, optimizer)
        vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader,   None)
        scheduler.step(vl_loss)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"    ep{epoch:>3} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} "
                  f"| vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} vl_f1={vl_f1:.4f}")

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_val_f1   = vl_f1
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PAT:
                if verbose:
                    print(f"    ⏹ Early stop at epoch {epoch} (patience={EARLY_STOP_PAT})")
                break

    return best_val_loss, best_val_f1, best_state, model


## 6. Seed 루프

In [6]:
# ══════════════════════════════════════════════════════
# 6. Seed 루프
# ══════════════════════════════════════════════════════
all_combos   = list(product(LR_GRID, DROPOUT_GRID, DMODEL_GRID, BATCH_GRID))
seed_results = []

for seed in SEED_LIST:
    print(f"\n{'═'*65}")
    print(f"  SEED {seed}")
    print(f"{'═'*65}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_all, y_all, test_size=0.30, random_state=seed, stratify=y_all
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )
    print(f"  Train:{len(X_train):,} Val:{len(X_val):,} Test:{len(X_test):,}")

    # ── Grid Search ──────────────────────────────────────
    print(f"\n  {'No':>4} | {'LR':>7} | {'Drop':>5} | {'dModel':>7} | {'Batch':>5} | "
          f"{'Val Loss':>9} | {'Val F1':>7}")
    print(f"  {'─'*62}")

    best_combo, best_vloss = None, float("inf")
    best_state_g, best_model_g = None, None

    for idx, (lr, dropout, d_model, batch) in enumerate(all_combos, 1):
        vl_loss, vl_f1, state, mdl = train_single(
            lr, dropout, d_model, batch, X_train, y_train, X_val, y_val
        )
        marker = " ★" if vl_loss < best_vloss else ""
        print(f"  {idx:>4} | {lr:>7.0e} | {dropout:>5.1f} | {d_model:>7} | {batch:>5} | "
              f"{vl_loss:>9.4f} | {vl_f1:>7.4f}{marker}")

        if vl_loss < best_vloss:
            best_vloss, best_combo = vl_loss, (lr, dropout, d_model, batch)
            best_state_g, best_model_g = state, mdl

    lr, dropout, d_model, batch = best_combo
    print(f"\n  ✅ Best → LR={lr:.0e} Drop={dropout} dModel={d_model} Batch={batch} "
          f"ValLoss={best_vloss:.4f}")

    # ── Best combo 재학습 ────────────────────────────────
    print(f"\n  Retraining best combo...")
    _, _, best_state_g, best_model_g = train_single(
        lr, dropout, d_model, batch, X_train, y_train, X_val, y_val, verbose=True
    )
    torch.save(best_state_g, f"best_hc_mlp_seed{seed}.pt")

    # ── Test 평가 ────────────────────────────────────────
    best_model_g.load_state_dict(best_state_g)
    best_model_g.eval()

    test_loader = DataLoader(HCDataset(X_test, y_test), batch_size=batch)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            probs = best_model_g(X_b.to(device)).squeeze().cpu().numpy()
            all_probs.extend(np.atleast_1d(probs).tolist())
            all_labels.extend(y_b.squeeze().cpu().tolist())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs >= 0.5).astype(int)

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)

    print(f"\n  ── TEST RESULTS (seed={seed}) ───────────────────────")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4,
                                target_names=[unique_labels[0], unique_labels[1]]))
    print(f"  Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    seed_results.append({
        "seed": seed,
        "best_lr": lr, "best_dropout": dropout,
        "best_d_model": d_model, "best_batch": batch,
        "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1, "roc_auc": auc,
    })



═════════════════════════════════════════════════════════════════
  SEED 42
═════════════════════════════════════════════════════════════════
  Train:14,000 Val:3,000 Test:3,000

    No |      LR |  Drop |  dModel | Batch |  Val Loss |  Val F1
  ──────────────────────────────────────────────────────────────
     1 |   3e-04 |   0.3 |     512 |    64 |    0.1768 |  0.9278 ★

  ✅ Best → LR=3e-04 Drop=0.3 dModel=512 Batch=64 ValLoss=0.1768

  Retraining best combo...
    ep  1 | tr_loss=0.2801 tr_acc=0.8815 | vl_loss=0.2459 vl_acc=0.9003 vl_f1=0.8951
    ep  5 | tr_loss=0.2198 tr_acc=0.9108 | vl_loss=0.1928 vl_acc=0.9227 vl_f1=0.9241
    ep 10 | tr_loss=0.2129 tr_acc=0.9136 | vl_loss=0.1830 vl_acc=0.9263 vl_f1=0.9254
    ep 15 | tr_loss=0.2061 tr_acc=0.9116 | vl_loss=0.1916 vl_acc=0.9240 vl_f1=0.9222
    ep 20 | tr_loss=0.1969 tr_acc=0.9226 | vl_loss=0.1811 vl_acc=0.9247 vl_f1=0.9248
    ep 25 | tr_loss=0.1887 tr_acc=0.9241 | vl_loss=0.1768 vl_acc=0.9280 vl_f1=0.9290
    ep 30 | tr_loss=

## 7. 요약

In [7]:
# ══════════════════════════════════════════════════════
# 7. 전체 Seed 요약
# ══════════════════════════════════════════════════════
print(f"\n{'═'*75}")
print("  SUMMARY ACROSS SEEDS")
print(f"{'═'*75}")

summary_df = pd.DataFrame(seed_results)
print(f"\n  {'Seed':>5} | {'Accuracy':>9} | {'Precision':>10} | {'Recall':>8} | "
      f"{'F1':>8} | {'ROC-AUC':>8} | Best Combo")
print(f"  {'─'*80}")
for _, row in summary_df.iterrows():
    print(f"  {int(row.seed):>5} | {row.accuracy:>9.4f} | {row.precision:>10.4f} | "
          f"{row.recall:>8.4f} | {row.f1:>8.4f} | {row.roc_auc:>8.4f} | "
          f"lr={row.best_lr:.0e} do={row.best_dropout} dm={int(row.best_d_model)} bs={int(row.best_batch)}")

print(f"  {'─'*80}")
print(f"  {'Mean':>5} | {summary_df.accuracy.mean():>9.4f} | "
      f"{summary_df.precision.mean():>10.4f} | {summary_df.recall.mean():>8.4f} | "
      f"{summary_df.f1.mean():>8.4f} | {summary_df.roc_auc.mean():>8.4f}")
print(f"  {'Std':>5} | {summary_df.accuracy.std():>9.4f} | "
      f"{summary_df.precision.std():>10.4f} | {summary_df.recall.std():>8.4f} | "
      f"{summary_df.f1.std():>8.4f} | {summary_df.roc_auc.std():>8.4f}")

summary_df.to_csv("seed_results_summary.csv", index=False)
print(f"\n  Saved → seed_results_summary.csv")
print("\nDone!")


═══════════════════════════════════════════════════════════════════════════
  SUMMARY ACROSS SEEDS
═══════════════════════════════════════════════════════════════════════════

   Seed |  Accuracy |  Precision |   Recall |       F1 |  ROC-AUC | Best Combo
  ────────────────────────────────────────────────────────────────────────────────
     42 |    0.9163 |     0.9217 |   0.9100 |   0.9158 |   0.9762 | lr=3e-04 do=0.3 dm=512 bs=64
     43 |    0.9293 |     0.9226 |   0.9373 |   0.9299 |   0.9804 | lr=3e-04 do=0.3 dm=512 bs=64
     44 |    0.9197 |     0.9166 |   0.9233 |   0.9200 |   0.9772 | lr=3e-04 do=0.3 dm=512 bs=64
     45 |    0.9197 |     0.9128 |   0.9280 |   0.9203 |   0.9778 | lr=3e-04 do=0.3 dm=512 bs=64
     46 |    0.9237 |     0.9217 |   0.9260 |   0.9238 |   0.9782 | lr=3e-04 do=0.3 dm=512 bs=64
  ────────────────────────────────────────────────────────────────────────────────
   Mean |    0.9217 |     0.9191 |   0.9249 |   0.9220 |   0.9780
    Std |    0.0050 |     0

PermissionError: [Errno 13] Permission denied: 'seed_results_summary.csv'